# Intro

## Loading

In [ ]:
from pathlib import Path
import pandas as pd

# Find the project root without importing pipeline.constants,
# which also creates a Spark session.
project_root = next(
    path
    for path in (Path.cwd(), *Path.cwd().parents)
    if (path / "pipeline" / "constants.py").is_file()
)

## Find requested runs

In [ ]:
# Load each Excel report, keyed by its scale factor.
reports = {}
for path in sorted((project_root / "data").glob("*.xlsx")):
    sheets = pd.read_excel(path, sheet_name=None)

    scale_values = {
        float(value)
        for sheet_name in ("Query Runs", "Run Summary")
        for value in sheets[sheet_name]["scale_factor"].dropna().unique()
    }
    if len(scale_values) != 1:
        raise ValueError(
            f"{path.name} should contain one consistent scale factor; "
            f"found {scale_values}"
        )

    scale_factor = scale_values.pop()
    key = f"{scale_factor:g}GB"
    if key in reports:
        raise ValueError(f"More than one workbook found for scale factor {key}")

    reports[key] = sheets

if not reports:
    raise FileNotFoundError(f"No Excel reports found in {project_root / 'data'}")

sorted(reports)


In [ ]:
# Combine the run summaries from all scale-factor workbooks.
run_catalog = pd.concat(
    [sheets["Run Summary"] for sheets in reports.values()],
    ignore_index=True,
)

run_catalog.head()

In [ ]:
# Filter the desired parameters to analyse
selected_runs = run_catalog[
    # (run_catalog["scale_factor"] == 1)
    # & 
    # (run_catalog["code_version"] == "18d9570")
]


display(selected_runs.head())

In [ ]:
print("Dict report keys:", list(reports.keys()))
print("Sheets for scale factor: ", list(reports["1GB"].keys()))
print("Columns inside Run Summary sheet:", list(reports["1GB"]["Run Summary"].columns))

In [ ]:
# Combine query rows from each workbook, then keep only selected runs.
query_runs = pd.concat(
    [sheets["Query Runs"] for sheets in reports.values()],
    ignore_index=True,
)
query_runs = query_runs[query_runs["run_id"].isin(selected_runs["run_id"])]
display(query_runs.sort_values(by="query_id").head())

In [ ]:
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns

# ---------------------------------------------------------
# 1. Consolidate 'Query Runs' data across all scale factors
# ---------------------------------------------------------
all_runs = []
sf_map = {"1GB": 1, "2GB": 2, "5GB": 5, "10GB": 10}

for sf_name, sf_numeric in sf_map.items():
    if sf_name in reports and "Query Runs" in reports[sf_name]:
        df_run = reports[sf_name]["Query Runs"].copy()
        df_run["sf_numeric"] = sf_numeric
        all_runs.append(df_run)

df_all = pd.concat(all_runs, ignore_index=True)

# Filter for successful executions only
df_success = df_all[df_all["status"] == "success"].copy()

# Aggregate multi-statement queries to compute total execution time per query
query_totals = (
    df_success.groupby(["query_id", "query_file", "sf_numeric"])["avg_seconds"]
    .sum()
    .reset_index()
)

# Pivot table: Rows = query_id, Columns = scale factor (1, 2, 5, 10)
pivot_times = query_totals.pivot(
    index=["query_id", "query_file"], columns="sf_numeric", values="avg_seconds"
).reset_index()

# ---------------------------------------------------------
# 2. Compute bottleneck metrics to rank critical queries
# ---------------------------------------------------------
pivot_times["time_1gb"] = pivot_times[1]
pivot_times["time_10gb"] = pivot_times[10]

# Scaling ratio (ideal linear scaling = 10.0x for a 1GB to 10GB scale change)
pivot_times["scaling_ratio"] = pivot_times["time_10gb"] / pivot_times["time_1gb"]

# Min-max normalization for runtime and scaling degradation
norm_time = (pivot_times["time_10gb"] - pivot_times["time_10gb"].min()) / (
    pivot_times["time_10gb"].max() - pivot_times["time_10gb"].min()
)
norm_scaling = (
    pivot_times["scaling_ratio"] - pivot_times["scaling_ratio"].min()
) / (pivot_times["scaling_ratio"].max() - pivot_times["scaling_ratio"].min())

# Composite bottleneck score (60% weight on absolute runtime, 40% on scaling degradation)
pivot_times["bottleneck_score"] = (0.6 * norm_time) + (0.4 * norm_scaling)

# Rank and select Top 10 candidate queries for targeted optimization
top10_candidates = pivot_times.sort_values(
    by="bottleneck_score", ascending=False
).head(10)

print("\n=== TOP 10 CANDIDATE QUERIES FOR TARGETED OPTIMIZATION ===")
display_cols = [
    "query_file",
    "time_1gb",
    "time_10gb",
    "scaling_ratio",
    "bottleneck_score",
]
print(top10_candidates[display_cols].to_string(index=False))

# ---------------------------------------------------------
# 3. Figure 1: Scatter Plot (Runtime at 10GB vs. Scaling Ratio)
# ---------------------------------------------------------
plt.figure(figsize=(10, 6))
sns.set_theme(style="whitegrid")

scatter = plt.scatter(
    pivot_times["scaling_ratio"],
    pivot_times["time_10gb"],
    c=pivot_times["bottleneck_score"],
    cmap="viridis",
    s=70,
    alpha=0.75,
    edgecolor="k",
)
cbar = plt.colorbar(scatter)
cbar.set_label("Bottleneck Score", fontsize=11)

# Theoretical linear scaling reference line (10x growth)
plt.axvline(
    x=10,
    color="crimson",
    linestyle="--",
    linewidth=1.5,
    alpha=0.8,
    label="Ideal Linear Scaling (10x)",
)

# Annotate the top 10 bottleneck queries
for _, row in top10_candidates.iterrows():
    plt.annotate(
        row["query_file"].replace(".sql", ""),
        (row["scaling_ratio"], row["time_10gb"]),
        xytext=(5, 5),
        textcoords="offset points",
        fontsize=9,
        fontweight="bold",
    )

plt.title("Query Bottleneck Diagnostic: Absolute Runtime vs. Scaling Degradation", fontsize=13)
plt.xlabel("Scaling Factor Ratio ($T_{10GB} / T_{1GB}$) [Lower is Better]", fontsize=11)
plt.ylabel("Execution Time at 10GB (seconds) [Lower is Better]", fontsize=11)
plt.yscale("log")
plt.legend(loc="upper left")
plt.tight_layout()
plt.show()

# ---------------------------------------------------------
# 4. Figure 2: Scalability Curves for the Selected Top 10
# ---------------------------------------------------------
plt.figure(figsize=(11, 7))

top_ids = top10_candidates["query_id"].tolist()
df_top_curves = query_totals[query_totals["query_id"].isin(top_ids)]

palette = sns.color_palette("tab10", len(top_ids))

for idx, q_id in enumerate(top_ids):
    q_data = df_top_curves[df_top_curves["query_id"] == q_id].sort_values("sf_numeric")
    q_name = q_data["query_file"].iloc[0].replace(".sql", "")
    plt.plot(
        q_data["sf_numeric"],
        q_data["avg_seconds"],
        marker="o",
        linewidth=2,
        label=q_name,
        color=palette[idx],
    )

plt.title("Scalability Progression Across Scale Factors (Top 10 Bottlenecks)", fontsize=13)
plt.xlabel("Scale Factor", fontsize=11)
plt.ylabel("Mean Execution Time (seconds, log scale)", fontsize=11)
plt.xticks([1, 2, 5, 10], labels=["1GB", "2GB", "5GB", "10GB"])
plt.yscale("log")
plt.legend(bbox_to_anchor=(1.05, 1), loc="upper left", title="Query")
plt.tight_layout()
plt.show()

# Analysis